# Fold 2a — Segmentación de imágenes con grafos: Normalized Cut y GrabCut

Reencuadre de la técnica hacia un caso de **inspección visual de calidad**: aislar la región de una pieza o producto que podría contener un defecto, separándola del fondo o de regiones irrelevantes de la imagen.

Dos enfoques:
1. **Superpíxeles (SLIC) + grafo de adyacencia (RAG) + Normalized Cut** — segmentación automática por similitud de color y proximidad espacial.
2. **GrabCut** — segmentación semiautomática fondo/objeto a partir de una caja delimitadora inicial, refinable con una máscara manual.

In [ ]:
# pip install --upgrade scikit-image opencv-python --break-system-packages
from skimage import data, segmentation, color
import matplotlib.pyplot as plt

# Imagen de ejemplo (sustituir por una imagen real de inspección de producto)
img = data.coffee()

plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title("Imagen de entrada")
plt.axis('off')
plt.show()

## 1. Superpíxeles con SLIC

SLIC agrupa píxeles vecinos y de color similar. `n_segments` y `compactness` se calibran según el tamaño esperado del defecto en una pieza real — no se dejan en el valor por defecto del ejemplo original.

In [ ]:
labels_slic = segmentation.slic(img, compactness=25, n_segments=250, start_label=1)
out_slic = color.label2rgb(labels_slic, img, kind='avg')

plt.figure(figsize=(4, 4))
plt.imshow(out_slic)
plt.title(f"Superpíxeles SLIC ({labels_slic.max()} regiones)")
plt.axis('off')
plt.show()

## 2. Grafo de adyacencia de regiones (RAG) + Normalized Cut

Se construye un grafo donde cada superpíxel es un nodo y las aristas conectan regiones vecinas, ponderadas por similitud de color. El corte normalizado busca la partición que minimiza la similitud *entre* grupos y maximiza la similitud *dentro* de cada grupo.

In [ ]:
try:
    from skimage.future import graph  # scikit-image < 0.23
except ImportError:
    from skimage import graph  # scikit-image >= 0.23

g = graph.rag_mean_color(img, labels_slic, mode='similarity')
labels_ncut = graph.cut_normalized(labels_slic, g)
out_ncut = color.label2rgb(labels_ncut, img, kind='avg')

fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(out_slic); ax[0].set_title("Superpíxeles (SLIC)"); ax[0].axis('off')
ax[1].imshow(out_ncut); ax[1].set_title("Corte normalizado (RAG + Ncut)"); ax[1].axis('off')
plt.tight_layout()
plt.show()

print(f"Regiones finales tras el corte normalizado: {len(set(labels_ncut.flatten()))}")

## 3. GrabCut: segmentación semiautomática fondo/objeto

Útil cuando se conoce aproximadamente dónde está el objeto (bounding box de una cámara de línea fija) y se necesita aislarlo del fondo con precisión de píxel.

In [ ]:
import numpy as np
import cv2

img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
mask = np.zeros(img_bgr.shape[:2], np.uint8)

bgdModel = np.zeros((1, 65), np.float64)
fgdModel = np.zeros((1, 65), np.float64)

# Rectángulo que aproxima la región del objeto de interés — ajustar a la pieza real
h, w = img_bgr.shape[:2]
rect = (int(w*0.1), int(h*0.1), int(w*0.8), int(h*0.8))

cv2.grabCut(img_bgr, mask, rect, bgdModel, fgdModel, 5, cv2.GC_INIT_WITH_RECT)

mask_binaria = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')
img_segmentada = img * mask_binaria[:, :, np.newaxis]

plt.figure(figsize=(4, 4))
plt.imshow(img_segmentada)
plt.title("GrabCut — objeto aislado")
plt.axis('off')
plt.show()

## 4. Ajuste documentado: máscara manual cuando el corte automático falla

En bordes finos o de baja diferencia de color con el fondo, el corte automático de GrabCut deja regiones sin segmentar correctamente. La corrección es una segunda pasada con `cv2.GC_INIT_WITH_MASK`, usando una máscara manual (o generada por un umbral adicional) para las zonas problemáticas:

```python
# newmask: imagen binaria donde 255 = seguro objeto, 0 = seguro fondo, resto sin marcar
mask[newmask == 0] = 0
mask[newmask == 255] = 1

mask, bgdModel, fgdModel = cv2.grabCut(
    img_bgr, mask, None, bgdModel, fgdModel, 5, cv2.GC_INIT_WITH_MASK
)
```

Esta segunda pasada mejora notablemente el resultado en zonas de borde ambiguo, a costa de requerir una anotación adicional (parcial) del caso difícil.

## 5. Hacia la aplicación real

- La salida de este notebook (máscara binaria del objeto o de la región de interés) es el insumo para medir **tamaño, forma o color del defecto** con reglas simples, o para recortar la región antes de pasarla a un clasificador de defectos.
- El siguiente notebook (`03_generacion_gan_sinteticos.ipynb`) aborda el problema complementario: qué hacer cuando la clase de defecto tiene muy pocos ejemplos reales para entrenar ese clasificador.